<a href="https://colab.research.google.com/github/GustavoFA/IA368/blob/main/notebooks/9_ReAct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IA368

## ReAct - Reasoning and Acting


Gustavo Freitas Alves

236249

---
## Imports

In [ ]:
!pip install -U -q langchain-community faiss-cpu langgraph langsmith langchain-openai bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.0/396.0 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.6/207.6 kB 18.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sour

In [ ]:
import os
import json
import requests
import numpy as np
from tqdm import tqdm
from bert_score import score
from google.colab import files
from langgraph.prebuilt import ToolNode
from langchain.chat_models import init_chat_model
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, MessagesState, END
from langchain_community.vectorstores import FAISS as faiss
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

---
## Dados

#### Questões

Obtendo as questões

In [ ]:
questions_link = r'https://huggingface.co/datasets/unicamp-dl/BR-TaxQA-R/resolve/main/questions_QA_2024_v1.0.json'
questions_request = requests.get(questions_link)
questions = questions_request.json()

#### Documentos

Para o caso dos documentos fiz o processamento em outro notebook (para deixar esse mais limpo e focado na parte do agente).

[Notebook processamento](https://colab.research.google.com/drive/1NktQsqtRTwsnH1MnhGsqSmniNaoZLTWc?usp=sharing)

Obtendo os textos indexados

In [ ]:
%%time
uploaded = files.upload()

Saving index.pkl to index.pkl
Saving index.faiss to index.faiss
CPU times: user 11.6 s, sys: 19.1 s, total: 30.7 s
Wall time: 4min 48s


In [ ]:
!mkdir faiss_taxqa
!mv index.faiss index.pkl faiss_taxqa/

In [ ]:
%%capture
if 'model' not in globals() or 'index' not in globals():
  model = HuggingFaceEmbeddings(
        model_name = "neuralmind/bert-base-portuguese-cased",
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True}
  )

  if not os.path.exists('faiss_taxqa'):
    raise('Import faiss files (.faiss and .pkl)')

  index = faiss.load_local("faiss_taxqa", model, allow_dangerous_deserialization=True)

Função para pesquisa dos documentos utilizando FAISS

In [ ]:
def search_docs(query:str, index=index, model=model, k:int=3, verbose:bool=False) -> str:
  # procura dos textos semanticamente próximos
  near_index = index.similarity_search(query, k)
  # resposta final
  context = "\n\n".join(
      [f"informação {i+1}: {doc.page_content}" for i, doc in enumerate(near_index)]
  )
  if verbose:
    print(context)
  return context

Verificando o funcionamento

In [ ]:
search_docs("Quem está obrigado a apresentar a Declaração de Ajuste Anual relativa ao exercício de 2024, ano-calendário de 2023?", verbose=True)

informação 1: . 36 em relação às participações em entidades controladas detidas em 31 de dezembro de 2023 deverá: I - indicar a sua opção na DAA a ser entregue em 2024, dentro do prazo, relativa ao ano-calendário de 2023, para produzir efeitos a partir de 1º de janeiro de 2024

informação 2: . DECLARA, em caráter normativo, às Superintendências Regionais da Receita Federal e aos demais interessados que: 1. O Demonstrativo da Apuração dos Ganhos de Capital de que trata o subitem 8.3 da IN RF 19/91, a ser anexado à Declaração de Rendimentos do exercício de 1991, é o do ano-base de 1989, exercício de 1990, adaptado à legislação tributária em vigor no ano-base de 1990. 1.1 Na apuração do ganho de capital na alienação de bens comuns, decorrentes do regime de casamento, o limite de 10.000 BTN aplica-se em relação ao bem. 2

informação 3: . 5º no momento em que os recursos se tornarem disponíveis no País. (Incluído(a) pelo(a) Instrução Normativa RFB nº 1654, de 27 de julho de 2016) Seção IV D

'informação 1: . 36 em relação às participações em entidades controladas detidas em 31 de dezembro de 2023 deverá: I - indicar a sua opção na DAA a ser entregue em 2024, dentro do prazo, relativa ao ano-calendário de 2023, para produzir efeitos a partir de 1º de janeiro de 2024\n\ninformação 2: . DECLARA, em caráter normativo, às Superintendências Regionais da Receita Federal e aos demais interessados que: 1. O Demonstrativo da Apuração dos Ganhos de Capital de que trata o subitem 8.3 da IN RF 19/91, a ser anexado à Declaração de Rendimentos do exercício de 1991, é o do ano-base de 1989, exercício de 1990, adaptado à legislação tributária em vigor no ano-base de 1990. 1.1 Na apuração do ganho de capital na alienação de bens comuns, decorrentes do regime de casamento, o limite de 10.000 BTN aplica-se em relação ao bem. 2\n\ninformação 3: . 5º no momento em que os recursos se tornarem disponíveis no País. (Incluído(a) pelo(a) Instrução Normativa RFB nº 1654, de 27 de julho de 2016) Seção

---
## Agente

#### Modelo de LLM

Neste agente utilizo o modelo gpt-5-nano.

In [ ]:
# Insert here your key
KEY = ''
os.environ['OPENAI_API_KEY'] = KEY

Especifico para o modelo responder em formato JSON

In [ ]:
llm = init_chat_model(
    "openai:gpt-5-nano",
    # resposta em formato json
    model_kwargs={"response_format": {"type": "json_object"}}
)

### Nós

Nó inicial, com mensagem de introdução para o modelo. Aqui vale ressaltar que como estou trabalhando com um dataset em português-BR, logo, todo tipo de comunicação que eu estabelecer com o modelo será nessa linguagem. Lembrando que o modelo é multi línguas.

In [ ]:
sys_msg = SystemMessage(content=r"""
Você é um assistente especializado no Imposto de Renda de Pessoas Físicas (IRPF).

Seu objetivo é responder perguntas sobre o IRPF com base em documentos oficiais da Receita Federal.
Caso não encontre a resposta no contexto disponível, siga estas instruções:

1. Quando não houver contexto suficiente:
   - Indique claramente que não foi possível encontrar a resposta.
   - Utilize a ferramenta de busca para recuperar informações nos documentos oficiais.

2. Quando houver contexto suficiente:
   - Forneça a resposta final de forma clara, objetiva e precisa.
   - Sempre que possível, cite a fonte ou base legal (por exemplo: "Instrução Normativa nº ...", "Lei nº ...").

3. Formato obrigatório de saída (sempre em JSON válido):
   ```json
   {
     "action": "busca" ou "resposta",
     "query": "pergunta relevante ou termos de busca",
     "answer": "resposta final adaptada ao contexto (se houver)"
   }
"""
)

Nó de raciocínio

In [ ]:
def reasoning_node(state: MessagesState):
  """
    Nó de raciocínio do modelo
  """
  messages = [sys_msg] + state["messages"]
  print(f'[MESSAGE] : {messages}')
  response = llm.invoke(messages)
  print(f'[RESPONSE] : {response}')
  return {"messages": [response]}

Nó da ação de pesquisar

In [ ]:
def search_node(state: MessagesState, verbose:bool=False):
  """
    Nó da ação de pesquisar nos documentos
  """
  last_msg = state["messages"][-1]
  try:
      parsed = json.loads(last_msg.content)
  except Exception:
      parsed = {}
  query = parsed.get("query", "")
  # textos mais pertinentes
  results = search_docs(query)
  print(f'[SEARCH RESULTS] : {results}')
  return {"messages": [AIMessage(content=f"Contexto buscado :\n{results}")]}

Laço do agente

Máximo de ações de pesquisa

In [ ]:
MAX_SEARCHS = 2

Função para controle do fluxo do agente

In [ ]:
def router(state: MessagesState, max_searchs: int = MAX_SEARCHS, max_steps: int = MAX_SEARCHS+2):
  """
    Controle de fluxo do ReAct
  """

  # contador de pesquisas ### SEMRPRE ZERADA
  # global n_searchs, n_steps

  # # Estado inicial - ### SEMPRE REINICIA
  # if (len(state)<=1):
  #     n_searchs = 0
  #     n_steps = 0

  ########## MEGA PROBLEMA - NÃO CONSIGO FAZER UMA VARIÁVEL PARA CONTABILIZAR NÚMERO DE PASSOS
  ## OU MESMO NÚMERO DE PESQUISAS.
  if "n_searchs" not in state:
    state["n_searchs"] = 0
  if "n_steps" not in state:
    state["n_steps"] = 0

  # Controle do número de passos (PROBLEMA: meu agente está em loop infinito o tempo todo)
  # n_steps += 1
  state["n_steps"] += 1
  if state["n_steps"] > max_steps:
    print("Finalização forçada - agente atingiu o número máximo de passos")
    return END

  # obter última mensagem
  last_msg = state["messages"][-1]
  try:
      parsed = json.loads(last_msg.content)
  except Exception:
      parsed = {}

  # verificação da ação
  action = parsed.get("action", "não achou a ação").lower()

  # impressão para debug
  print(f'[ACTION] - passo {state["n_steps"]} : {action} | [N_SEARCH] : {state["n_searchs"]}')


  if action == "busca":
    if state['n_searchs'] < max_searchs:
      state['n_searchs'] += 1
      return 'act'
    else:
      print('Finalização forçada - agente atingiu máximo de buscas')
      return END

  elif action == "resposta":
    return END

  else:
    return 'rsn'

Construção do Grafo - fluxo de raciocínio do agente

In [ ]:
# construtor de grafo de estados (baseado no MessagesState = dict do histórico de mensagens)
builder = StateGraph(MessagesState)

# Adicionando os nós
builder.add_node("rsn", reasoning_node)
builder.add_node("act", search_node)

# Condições para nó de raciocínio
builder.add_conditional_edges("rsn", router)

# Fluxo: BUSCAR (acting) ---> RACIOCINAR (reasoning)
builder.add_edge("act", "rsn")

# Ponto inicial
builder.set_entry_point("rsn")

# Contrução do grafo (agente)
agent = builder.compile()

Resposta final do modelo

In [ ]:
def get_final_answer(result):
  """
    Extrai a resposta final do resultado do agente LangGraph.
  """
  for msg in reversed(result.get("messages", [])):
    if isinstance(msg, AIMessage):
      try:
       data = json.loads(msg.content)
      except json.JSONDecodeError:
        continue  # pula mensagens mal formatadas

      # Obter resposta final
      if data.get("action") == "resposta":
        return data.get("answer", "Nenhuma resposta encontrada")

  return "Nenhuma resposta final foi identificada no resultado."

Teste do agente

In [ ]:
%%time
q = "Quem está obrigado a apresentar a Declaração de Ajuste Anual relativa ao exercício de 2024, ano-calendário de 2023?"
result = agent.invoke({"messages": [HumanMessage(content=q)]})

[MESSAGE] : [SystemMessage(content='\nVocê é um assistente especializado no Imposto de Renda de Pessoas Físicas (IRPF).\n\nSeu objetivo é responder perguntas sobre o IRPF com base em documentos oficiais da Receita Federal.\nCaso não encontre a resposta no contexto disponível, siga estas instruções:\n\n1. Quando não houver contexto suficiente:\n   - Indique claramente que não foi possível encontrar a resposta.\n   - Utilize a ferramenta de busca para recuperar informações nos documentos oficiais.\n\n2. Quando houver contexto suficiente:\n   - Forneça a resposta final de forma clara, objetiva e precisa.\n   - Sempre que possível, cite a fonte ou base legal (por exemplo: "Instrução Normativa nº ...", "Lei nº ...").\n\n3. Formato obrigatório de saída (sempre em JSON válido):\n   ```json\n   {\n     "action": "busca" ou "resposta",\n     "query": "pergunta relevante ou termos de busca",\n     "answer": "resposta final adaptada ao contexto (se houver)"\n   }\n', additional_kwargs={}, respons

In [ ]:
result

{'messages': [HumanMessage(content='Quem está obrigado a apresentar a Declaração de Ajuste Anual relativa ao exercício de 2024, ano-calendário de 2023?', additional_kwargs={}, response_metadata={}, id='b251919e-0e1b-4716-bc70-0ff1937c4e20'),
  AIMessage(content='{\n  "action": "resposta",\n  "query": "Quem está obrigado a apresentar a Declaração de Ajuste Anual relativa ao exercício de 2024, ano-calendário de 2023?",\n  "answer": "Está obrigado a apresentar a Declaração de Ajuste Anual do IRPF 2024 (referente ao ano-calendário de 2023) a pessoa física residente no Brasil que, em 31/12/2023, se enquadrou em pelo menos uma das situações a seguir: 1) recebeu rendimentos tributáveis cuja soma foi superior a R$ 28.559,70 em 2023; 2) recebeu rendimentos isentos, não tributáveis ou tributados exclusivamente na fonte cuja soma foi superior a R$ 40.000,00 em 2023; 3) realizou ganhos de capital na alienação de bens ou direitos sujeitos à incidência do imposto, ou realizou operações em bolsa de v

In [ ]:
get_final_answer(result)

'Está obrigado a apresentar a Declaração de Ajuste Anual do IRPF 2024 (referente ao ano-calendário de 2023) a pessoa física residente no Brasil que, em 31/12/2023, se enquadrou em pelo menos uma das situações a seguir: 1) recebeu rendimentos tributáveis cuja soma foi superior a R$ 28.559,70 em 2023; 2) recebeu rendimentos isentos, não tributáveis ou tributados exclusivamente na fonte cuja soma foi superior a R$ 40.000,00 em 2023; 3) realizou ganhos de capital na alienação de bens ou direitos sujeitos à incidência do imposto, ou realizou operações em bolsa de valores, mercadorias, futuros, etc., com ganho líquido; 4) possuía bens e direitos, em 31/12/2023, com valor total superior a R$ 300.000,00; 5) tornou-se residente no Brasil em 2023 e permaneceu nessa condição até 31/12/2023. Observação: há situações adicionais previstas pela norma (por exemplo rendimentos de fontes no exterior, dependentes, etc.). A lista acima resume as principais condições; para confirmação detalhada, consulte a

#### Executando agente completo

In [ ]:
def react_exe(questions:list, agent=agent, max_q:int=100) -> dict:
  """
    Função geral do agente com armazenamento das respostas e ações
  """
  # salvar questão, ações e resposta final
  results = []
  cont = 0
  for question in tqdm(questions[:max_q], desc="Executando perguntas", ncols=100):
      # pergunta
      question = question['question_text']
      cont += 1
      print(f'\nQUESTÃO {cont} : {question}')

      # Executar o grafo (raciocínio + ações)
      result = agent.invoke({"messages": [HumanMessage(content=question)]})

      # Obter todas as mensagens (histórico do ciclo ReAct)
      messages = result.get("messages", [])

      # Inicializar registro da execução
      record = {
          "question": question,
          "actions": [],
          "final_answer": None
      }

      # Percorrer mensagens e extrair dados
      for msg in messages:
          if isinstance(msg, AIMessage):
              try:
                  data = json.loads(msg.content)
              except json.JSONDecodeError:
                  continue

              # Registrar cada ação intermediária (busca, reasoning, etc.)
              if "action" in data and data["action"] == "busca":
                  record["actions"].append({
                      "action": "busca",
                      "query": data.get("query", "")
                  })

              # Capturar a resposta final
              if "action" in data and data["action"] == "resposta":
                  record["final_answer"] = data.get("answer", "")

      results.append(record)

  return results

In [ ]:
MAX_QUESTIONS = 5

In [ ]:
%%time
results = react_exe(questions, max_q=MAX_QUESTIONS)

Executando perguntas:   0%|                                                   | 0/5 [00:00<?, ?it/s]


QUESTÃO 1 : Quem está obrigado a apresentar a Declaração de Ajuste Anual relativa ao exercício de 2024, ano-calendário de 2023?
[MESSAGE] : [SystemMessage(content='\nVocê é um assistente especializado no Imposto de Renda de Pessoas Físicas (IRPF).\n\nSeu objetivo é responder perguntas sobre o IRPF com base em documentos oficiais da Receita Federal.\nCaso não encontre a resposta no contexto disponível, siga estas instruções:\n\n1. Quando não houver contexto suficiente:\n   - Indique claramente que não foi possível encontrar a resposta.\n   - Utilize a ferramenta de busca para recuperar informações nos documentos oficiais.\n\n2. Quando houver contexto suficiente:\n   - Forneça a resposta final de forma clara, objetiva e precisa.\n   - Sempre que possível, cite a fonte ou base legal (por exemplo: "Instrução Normativa nº ...", "Lei nº ...").\n\n3. Formato obrigatório de saída (sempre em JSON válido):\n   ```json\n   {\n     "action": "busca" ou "resposta",\n     "query": "pergunta relevan

Executando perguntas:  20%|████████▌                                  | 1/5 [00:32<02:11, 32.82s/it]

[RESPONSE] : content='{\n  "action": "resposta",\n  "query": "Quem está obrigado a apresentar a Declaração de Ajuste Anual relativa ao exercício de 2024, ano-calendário de 2023?",\n  "answer": "Estão obrigados a apresentar a Declaração de Ajuste Anual do IRPF 2024 (ano-base 2023) os contribuintes pessoa física que, no ano-calendário de 2023, atenderem a pelo menos uma das seguintes situações: (1) terem tido rendimentos tributáveis cuja soma foi superior a R$ 28.559,70; (2) terem tido rendimentos isentos, não tributáveis ou tributados exclusivamente na fonte cuja soma foi superior a R$ 40.000,00; (3) ganho de capital na alienação de bens ou direitos sujeito à incidência do imposto; (4) realizado operações em bolsa de valores, mercadorias, futuros ou similares com ganho de capital; (5) possuídos bens e direitos com valor total superior a R$ 300.000,00; (6) auferido receita bruta de atividade rural acima de R$ 142.547,25. Caso nenhuma dessas situações se aplique, a declaração pode não ser

Executando perguntas:  40%|█████████████████▏                         | 2/5 [01:03<01:34, 31.63s/it]

...

[RESPONSE] : content='{\n  "action": "resposta",\n  "query": "Contribuinte titular ou sócio de empresa obrigado a apresentar a Declaração de Ajuste Anual exercício 2024 IRPF",\n  "answer": "Não. Ser titular ou sócio de uma empresa não implica automaticamente obrigação de apresentar a Declaração de Ajuste Anual do IRPF para o exercício de 2024. A obrigatoriedade depende de critérios estabelecidos pela Receita Federal, como: rendimentos tributáveis recebidos no ano-calendário (submetidos à tributação) acima do limite de isenção; rendimentos isentos/tributados na fonte cuja soma exceda esse limite; possuir bens e direitos com valor total superior a determinada faixa; ganhos de capital; atividade rural; entre outros itens listados pela RF. Além disso, se o titular/sócio recebeu pró-labore, distribuição de lucros ou outros rendimentos que integram a base de tributação, esses rendimentos podem tornar obrigatório o envio da declaração. Em caso de dúvidas, consulte as regras oficiais da Receit

Executando perguntas:  80%|██████████████████████████████████▍        | 4/5 [03:38<01:02, 62.66s/it]

[RESPONSE] : content='{\n  "action": "resposta",\n  "query": "Contribuinte que participou de quadro societário de sociedade anônima ou foi associado de cooperativa em 2023 precisa apresentar a Declaração de Ajuste Anual do IRPF 2024?",\n  "answer": "Não. a participação em quadro societário de uma sociedade anônima ou a associação a uma cooperativa em 2023 não, por si só, cria obrigação automática de entregar a Declaração de Ajuste Anual do IRPF referente ao exercício de 2024. A obrigatoriedade depende do conjunto de rendimentos auferidos no ano-base 2023 (rendimentos tributáveis, rendimentos isentos/não tributáveis, ganhos de capital) e de outros critérios previstos pela Receita Federal (por exemplo, posse de bens/direitos de valor acima de determinado limite, retenção de IR na fonte, etc.). Se, no total do ano-base, você teve rendimentos tributáveis acima do limite de isenção ou possuía bens acima do patamar estabelecido, a entrega é obrigatória; caso contrário, pode não ser. Para con

Executando perguntas: 100%|███████████████████████████████████████████| 5/5 [04:36<00:00, 55.35s/it]

[RESPONSE] : content='{\n  "action": "resposta",\n  "query": "Contribuinte responsável perante a RFB por CNPJ de associações (bairros, creches, clubes) em 2023 deve apresentar a Declaração de Ajuste Anual 2024?",\n  "answer": "Não. A obrigação de entregar a Declaração de Ajuste Anual do IRPF não decorre automaticamente do fato de a pessoa ser responsável perante a RFB por um CNPJ de uma associação. A obrigatoriedade está vinculada a rendimentos auferidos pela pessoa física no ano-base (rendimentos tributáveis, rendimentos com imposto retido na fonte, ganhos de capital, etc.) ou a outras situações previstas na legislação. Se o contribuinte não teve rendimentos tributáveis em 2023 e não se enquadra em nenhuma outra hipótese de obrigatoriedade, ele não precisa apresentar a Declaração de Ajuste Anual em 2024. Em caso de dúvida, verifique as regras específicas da IRPF 2024 na legislação da Receita Federal (Instruções Normativas/Lei aplicável) ou consulte o portal do IRPF para confirmar a su

---
## Avaliação

Obtendo as respostas de referência

In [ ]:
refs = ["".join(text for text in question['answer_cleaned']) for question in questions[:MAX_QUESTIONS]]

Obtendo as respostas geradas pelo modelo (apenas a final)

In [ ]:
preds = [result['final_answer'] for result in results]

Precisão, Recall e F1 score

In [ ]:
def compute_scores_bow(predicted_list, reference_list):
    """
      Calcula as métricas médias de precisão, recall e F1-Score (bag-of-words)
    """

    precisions, recalls, f1s = [], [], []

    for predicted, gold in zip(predicted_list, reference_list):
        # Casos vazios
        if not predicted or not gold:
            precisions.append(0.0)
            recalls.append(0.0)
            f1s.append(0.0)
            continue

        # Conjuntos de palavras (bag-of-words)
        pt = set(predicted.lower().split())
        gt = set(gold.lower().split())

        if not pt and not gt:
            precisions.append(1.0)
            recalls.append(1.0)
            f1s.append(1.0)
            continue
        if not pt or not gt:
            precisions.append(0.0)
            recalls.append(0.0)
            f1s.append(0.0)
            continue

        # Interseção
        common = pt & gt
        precision = len(common) / len(pt)
        recall = len(common) / len(gt)
        f1 = 0.0 if (precision + recall) == 0 else 2 * (precision * recall) / (precision + recall)

        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)

    return float(np.mean(precisions)), float(np.mean(recalls)), float(np.mean(f1s))

In [ ]:
prec, rec, f1 = compute_scores_bow(preds, refs)

In [ ]:
print(f'Métricas BOW:\nPrecisão: {prec:.2%}\nRecall: {rec:.2%}\nF1 Score: {f1:.2%}')

Métricas BOW:
Precisão: 24.92%
Recall: 35.45%
F1 Score: 24.55%


BERT score

In [ ]:
def compute_bert_score(predicted_list, reference_list, lang="pt"):
    """
    Calcula o BERTScore médio para uma lista de respostas preditas e de referência.
    Retorna um dicionário com Precision, Recall e F1 médios.
    """
    valid_pairs = [(p, r) for p, r in zip(predicted_list, reference_list) if p and r]
    if not valid_pairs:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    preds, refs = zip(*valid_pairs)
    P, R, F1 = score(list(preds), list(refs), lang=lang, verbose=False)

    return float(P.mean().cpu().item()), float(R.mean().cpu().item()), float(F1.mean().cpu().item())

In [ ]:
bert_prec, bert_rec, bert_f1 = compute_bert_score(preds, refs)

In [ ]:
print(f'Métricas BERT:\nPrecisão: {bert_prec:.2%}\nRecall: {bert_rec:.2%}\nF1 Score: {bert_f1:.2%}')

Métricas BERT:
Precisão: 68.62%
Recall: 72.47%
F1 Score: 70.41%


---
## Histórico completo

Deixo todo o histórico de mensagens trocadas com o agente (para analisar)

In [ ]:
for i, r in enumerate(results, 1):
  print("=" * 80)
  print(f" Questão {i}:")
  print(r['question'])
  print("\n Ações realizadas:")
  if r['actions']:
      for j, action in enumerate(r['actions'], 1):
          print(f"  {j}. ({action['action']}) -> {action['query']}")
  else:
      print("  Nenhuma ação registrada.")

  print("\n Resposta final:")
  print(r['final_answer'] or "Nenhuma resposta gerada.")

print("=" * 80)
print(f"TOTAL DE RESULTADOS EXIBIDOS: {len(results)}")

 Questão 1:
Quem está obrigado a apresentar a Declaração de Ajuste Anual relativa ao exercício de 2024, ano-calendário de 2023?

 Ações realizadas:
  Nenhuma ação registrada.

 Resposta final:
Estão obrigados a apresentar a Declaração de Ajuste Anual do IRPF 2024 (ano-base 2023) os contribuintes pessoa física que, no ano-calendário de 2023, atenderem a pelo menos uma das seguintes situações: (1) terem tido rendimentos tributáveis cuja soma foi superior a R$ 28.559,70; (2) terem tido rendimentos isentos, não tributáveis ou tributados exclusivamente na fonte cuja soma foi superior a R$ 40.000,00; (3) ganho de capital na alienação de bens ou direitos sujeito à incidência do imposto; (4) realizado operações em bolsa de valores, mercadorias, futuros ou similares com ganho de capital; (5) possuídos bens e direitos com valor total superior a R$ 300.000,00; (6) auferido receita bruta de atividade rural acima de R$ 142.547,25. Caso nenhuma dessas situações se aplique, a declaração pode não ser 